In [ ]:
X = df_train.drop(columns="régime_alimentaire")
y = df_train["régime_alimentaire"]

# One hot encoding categorical variables
ohe = OneHotEncoder(handle_unknown="ignore") # to handle cases where we might get unseen categories during test
X = ohe.fit_transform(X)

tree = DecisionTreeClassifier(criterion="gini",
                              max_depth=3,
                              random_state=RANDOM_STATE)

tree.fit(X, y)



plt.figure(figsize=(11, 10))

plot_tree(
    tree,
    feature_names=ohe.get_feature_names_out(),
    class_names=tree.classes_,
    filled=True,
    rounded=True,
    impurity=True,
    node_ids=True
)

plt.show()




X_test = df_test.drop(columns="régime_alimentaire")
X_test = ohe.transform(X_test)


from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

y_test = df_test["régime_alimentaire"]

y_pred = tree.predict(X_test)
leaf_ids = tree.apply(X_test)

results = df_test.copy()
results["prédiction"] = y_pred
results["correct"] = results["régime_alimentaire"] == results["prédiction"]
results["feuille"] = leaf_ids

display(results[[
    "régime_alimentaire",
    "prédiction",
    "correct",
    "feuille"
]])


ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=tree.classes_,
    cmap="Blues"
)

plt.title(f"Résultats sur le test set - accuracy = {accuracy_score(y_test, y_pred):.2f}")
plt.show()

print(classification_report(y_test, y_pred))



colors = results["correct"].map({True: "tab:green", False: "tab:red"})

plt.figure(figsize=(10, 4))
plt.scatter(results.index, results["feuille"], c=colors, s=120)

plt.xticks(rotation=90)
plt.ylabel("Feuille de l'arbre")
plt.title("Feuille atteinte par chaque dinosaure du test set")

for name, row in results.iterrows():
    plt.text(
        name,
        row["feuille"] + 0.05,
        row["prédiction"],
        rotation=90,
        ha="center",
        fontsize=8
    )

plt.show()
